In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
# UPDATE THESE PATHS to point to where you placed the dataset.
img_dir = "../data/bentham_lines/Images/Lines"
label_path = "../data/bentham_lines/labels.txt"

In [3]:
all_text = ""

with open(label_path, "r", encoding="utf-8") as f:
    for line in f:
        all_text += line.split("\t")[1]

CHARSET = sorted(set(all_text))

char_to_idx = {c:i+1 for i,c in enumerate(CHARSET)}
idx_to_char = {i+1:c for i,c in enumerate(CHARSET)}

print("Charset size:", len(CHARSET))

Charset size: 94


In [4]:
def preprocess_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError(f"Cannot load {img_path}")

    img = cv2.resize(img, (256, 64))

    img = img.astype(np.float32) / 255.0

    return img

In [5]:
class HTRDataset(Dataset):
    def __init__(self, img_dir, label_file):
        self.img_dir = img_dir
        self.samples = []

        with open(label_file, "r", encoding="utf-8") as f:
            for line in f:
                img, text = line.strip().split("\t")
                self.samples.append((img, text))

    def encode(self, text):
        return torch.tensor([char_to_idx[c] for c in text if c in char_to_idx], dtype=torch.long)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, text = self.samples[idx]

        img_path = os.path.join(self.img_dir, img_name)

        img = preprocess_image(img_path)

        img = torch.tensor(img).unsqueeze(0) 

        label = self.encode(text)

        return img, label

In [6]:
def collate_fn(batch):
    imgs = [b[0] for b in batch]
    labels = [b[1] for b in batch]

    imgs = torch.stack(imgs)

    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels = torch.cat(labels)

    return imgs, labels, label_lengths


dataset = HTRDataset(img_dir, label_path)

loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

In [7]:
class HTRModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU()
        )

        self.rnn = nn.LSTM(
            input_size=256,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)          

        x = torch.mean(x, dim=2)

        x = x.permute(0, 2, 1)   

        x, _ = self.rnn(x)

        x = self.fc(x)

        return x.log_softmax(2)

In [8]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

model = HTRModel(len(CHARSET) + 1).to(DEVICE)

ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

In [ ]:
for epoch in range(20):
    model.train()
    total_loss = 0

    for imgs, labels, label_lengths in loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to("cpu") 

        outputs = model(imgs)

       
        outputs_cpu = outputs.to("cpu")

        input_lengths = torch.full(
            (imgs.size(0),),
            outputs.size(1),
            dtype=torch.long
        )

        loss = ctc_loss(
            outputs_cpu.permute(1, 0, 2),  
            labels,
            input_lengths,
            label_lengths
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        total_loss += loss.item()/imgs.size(0)

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 741.1235
Epoch 2, Loss: 513.8381
Epoch 3, Loss: 508.1573
Epoch 4, Loss: 502.9061
Epoch 5, Loss: 497.7867
Epoch 6, Loss: 491.4418
Epoch 7, Loss: 484.7313
Epoch 8, Loss: 482.4999
Epoch 9, Loss: 475.5138
Epoch 10, Loss: 465.1337
Epoch 11, Loss: 453.2836
Epoch 12, Loss: 437.2374
Epoch 13, Loss: 426.0299
Epoch 14, Loss: 419.3879
Epoch 15, Loss: 409.1626
Epoch 16, Loss: 402.6349
Epoch 17, Loss: 395.8116
Epoch 18, Loss: 392.0254
Epoch 19, Loss: 386.5428
Epoch 20, Loss: 381.2066


In [10]:
def greedy_decode(output):
    output = output.argmax(2)

    decoded = []

    for seq in output:
        prev = -1
        text = ""

        for i in seq:
            i = i.item()

            if i != prev and i != 0:
                text += idx_to_char.get(i, "")

            prev = i

        decoded.append(text.strip())

    return decoded

In [11]:
model.eval()

imgs, labels, label_lengths = next(iter(loader))
imgs = imgs.to(DEVICE)

with torch.no_grad():
    outputs = model(imgs)

preds = greedy_decode(outputs)

gt_texts = []
start = 0

for length in label_lengths:
    length = length.item()
    
    text = ""
    for i in range(start, start + length):
        idx = labels[i].item()
        text += idx_to_char.get(idx, "")
    
    gt_texts.append(text)
    start += length

for i in range(5):
    print(f"GT   : {gt_texts[i]}")
    print(f"PRED : {preds[i]}")
    print("-" * 50)

GT   : which the Forms require to be charged as a Fact contradistinct
PRED : att th he anin t th thegt on the atities
--------------------------------------------------
GT   : " and the various and unforeseen misfortunes which had from time
PRED : at th amen and np rine anef ten tit an f rn tae
--------------------------------------------------
GT   : estimated and the punishment of it determined . But these
PRED : -ent t he prnemnt o th thtemn an the thee
--------------------------------------------------
GT   : -tion of Thomas Rhodes , and on the East , on part by a
PRED : the of hens thetin ane in the tait in pat o s
--------------------------------------------------
GT   : the Limits between these two Places in every Case that
PRED : th her tatieoen the he fes o any the thee
--------------------------------------------------


In [12]:
torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "epoch": epoch
}, "checkpoint.pth")

In [13]:
checkpoint = torch.load("checkpoint.pth", map_location=DEVICE)

model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])
start_epoch = checkpoint["epoch"]

In [14]:
import os
print(os.getcwd())

/Users/tejasjindal/Documents
